# Experimentation notebook for hybrid CNNs

## Dependencies

In [1]:
import os, sys, time, copy, argparse
from pathlib import Path

import numpy as np
import pandas as pd
import scipy as sp
import scipy.sparse as sps
import torch
import neuropythy as ny
import optuna

import matplotlib as mpl
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

import visual_autolabel as val

## Configuration

In [2]:
dataset2D_cache_path = '/data/visual-autolabel/datasets/HCP'
tx3Dto2D_cache_path = '/data/visual-autolabel/volumetric/data/tx3Dto2D'

val.image.dataset3D_cache_path = '/data/visual-autolabel/volumetric/data'
val.image.noddi_data_path = '/data/visual-autolabel/volumetric/NODDI'

inputs3D = ('graymask', 'T1', 'T2')
inputs2D = ('curvature', 'convexity', 'thickness')
outputs = ('V1', 'V2', 'V3')
num_epochs = 30
zoom = 1/2
subindex = (slice(2,-2), slice(8, 256+8), slice(2,-2))
dtype = torch.float32
device = 'cpu'
shuffle = True

## Setting up Training Loop

In [3]:
# Training and validation subjects
trn_sids = [100610, 118225, 140117, 158136, 197348, 214524, 346137, 412528,
            573249, 724446, 905147, 102311, 159239, 173334, 221319, 352738,
            429040, 725751, 826353, 910241, 102816, 145834, 162935, 175237,
            199655, 233326, 436845, 732243, 926862, 104416, 128935, 146129,
            164131, 200210, 365343, 463040, 751550, 859671, 927359, 105923,
            130114, 146432, 164636, 187345, 200311, 467351, 617748, 757764,
            942658, 108323, 130518, 165436, 191033, 200614, 249947, 381038,
            525541, 627549, 109123, 131217, 146937, 167036, 177746, 191336,
            201515, 385046, 536647, 638049, 770352, 872764, 111312, 167440,
            178142, 191841, 203418, 257845, 541943, 771354, 878776, 958976,
            111514, 132118, 169040, 178243, 393247, 547046, 654552, 878877,
            966975, 114823, 155938, 205220, 283543, 395756, 550439, 671855,
            783462, 898176, 971160, 156334, 180533, 193845, 318637, 397760,
            552241, 680957, 899885, 973770, 115825, 135124, 157336, 169747,
            181232, 401422, 562345, 690152, 814649, 901139, 995174, 116726,
            137128, 158035, 181636, 196144, 330324, 406836, 572045, 818859]
val_sids = [765864, 209228, 134829, 585256, 901442, 169444, 380036, 389357,
            581450, 198653, 115017, 782561, 176542, 246133, 185442, 601127,
            204521, 195041, 182739, 212419, 263436, 320826, 825048, 192641,
            360030, 177140, 146735, 126426, 789373, 871762, 172130, 171633]

In [4]:
trn_dataset3D = val.image.HCPDataset3D(
    sids=trn_sids,
    inputs=inputs3D,
    outputs=outputs,
    cache_path=val.image.dataset3D_cache_path,
    dtype=dtype,
    device=device,
    mkdir_mode=0o775,
    subindex=subindex,
    zoom=zoom)

In [5]:
trn_dataset2D = val.benson2025.hcp.HCPDataset(
    inputs2D, outputs,
    sids=trn_sids)

In [6]:
class VolumeToFlatImageDataset(torch.utils.data.Dataset):
    def __init__(self,
                 sids,
                 cache_path=tx3Dto2D_cache_path):
        self.cache_path = Path(cache_path)
        self.sids = sids
        self.data = {}
    def __len__(self):
        return len(self.sids)
    def __getitem__(self, k):
        sid = self.sids[k]
        if sid in self.data:
            return self.data[sid]
        matrix = torch.load(f"{self.cache_path}/{sid}.pt", weights_only=False)
        import scipy.sparse as sps
        (row, col, val) = sps.find(matrix)
        matrix = torch.sparse_coo_tensor(
            torch.as_tensor(np.array([row, col])), 
            torch.as_tensor(val),
            matrix.shape,
            dtype=torch.float32)
        #matrix = torch.tensor(matrix, dtype=torch.float32)
        self.data[sid] = matrix
        return matrix

In [7]:
class HCPHybridDataset(torch.utils.data.Dataset):
    def __init__(self,
                 sids,
                 inputs2D,
                 inputs3D,
                 outputs=('V1', 'V2', 'V3'),
                 cache_path2D=None,
                 cache_path3D=None,
                 transform_cache_path=tx3Dto2D_cache_path,
                 dtype=None,
                 device=None,
                 mkdir_mode=509,
                 subindex=(slice(2, -2, None), slice(8, 264, None), slice(2, -2, None)),
                 zoom=0.5,
                 ):
        self.transform_dataset = VolumeToFlatImageDataset(sids, cache_path=transform_cache_path)
        self.dataset3D = val.image.HCPDataset3D(
            sids=sids,
            inputs=inputs3D,
            outputs=outputs,
            cache_path=cache_path3D,
            dtype=dtype,
            device=device,
            mkdir_mode=0o775,
            subindex=subindex,
            zoom=zoom)
        self.dataset2D = val.benson2025.hcp.HCPDataset(
            inputs2D, 
            outputs,
            sids=sids,
            cache_path=cache_path2D)
        self.sids = sids
    def __len__(self):
        return len(self.sids)
    def __getitem__(self, k):
        inputdata3D, _ = self.dataset3D[k]
        transformdata3D = self.transform_dataset[k]
        inputdata2D, outputdata2D = self.dataset2D[k]
        return (inputdata3D, transformdata3D, inputdata2D, outputdata2D)
        

In [8]:
# Define custom collate function to handle sparse matrices
def hybrid_collate(batch):
    inputs3D, transforms, inputs2D, labels = zip(*batch)
    return (
        torch.stack(inputs3D),          # [B, C, D, H, W]
        list(transforms),              # keep sparse transforms as list
        torch.stack(inputs2D),         # [B, C, H, W]
        torch.stack(labels)            # [B, C, H, W]
    )

def objective(trial, inputs2D = inputs2D, inputs3D = inputs3D, outputs = outputs):
    
    lr = trial.suggest_float('lr', 0.001, 0.015)
    gamma = trial.suggest_float('gamma', 0.5, 1)
    #batch_size = trial.suggest_int('batch_size', 1, 10)
    batch_size = 1
    
    trn_dataset = HCPHybridDataset(
        sids=trn_sids,
        inputs2D=inputs2D,
        inputs3D=inputs3D,
        outputs=outputs,
        cache_path2D=dataset2D_cache_path,
        cache_path3D=val.image.dataset3D_cache_path,
        transform_cache_path=tx3Dto2D_cache_path,
        dtype=dtype,
        device=device,
        mkdir_mode=0o775,
        subindex=subindex,
        zoom=zoom
    )

    val_dataset = HCPHybridDataset(
        sids=val_sids,
        inputs2D=inputs2D,
        inputs3D=inputs3D,
        outputs=outputs,
        cache_path2D=dataset2D_cache_path,
        cache_path3D=val.image.dataset3D_cache_path,
        transform_cache_path=tx3Dto2D_cache_path,
        dtype=dtype,
        device=device,
        mkdir_mode=0o775,
        subindex=subindex,
        zoom=zoom
    )

    # Create DataLoaders with custom collate
    train_loader = DataLoader(trn_dataset, batch_size=batch_size, shuffle=True, collate_fn=hybrid_collate)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=hybrid_collate)
    dataloaders = {'trn': train_loader, 'val': val_loader}

    # Initialize model
    model = val.image.UNet(len(inputs3D), len(inputs3D), len(inputs2D), len(outputs))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=gamma)

    # Training loop
    best_loss = np.inf
    best_weights = None
    print("Starting training...")

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1:02d}/{num_epochs}")
        losses_epoch = {'trn': [], 'val': []}
        dice_losses_epoch = {'trn': [], 'val': []}
        wd = epoch / (num_epochs - 1)
        wb = 1 - wd

        for phase in ['trn', 'val']:
            print('  Training..' if phase == 'trn' else '  Testing..', end='')
            model.train() if phase == 'trn' else model.eval()
            running_loss = 0.0
            running_dice = 0.0
            count = 0

            for inputs3D, transforms, inputs2D, labels in dataloaders[phase]:
                inputs3D, inputs2D, labels = inputs3D.to(device), inputs2D.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'trn'):
                    outputs = model(inputs3D, inputs2D, transforms)
                    loss_bce = val.bce_loss(outputs, labels)
                    loss_dice = val.dice_loss(outputs, labels)
                    loss = wb * loss_bce + wd * loss_dice

                    if phase == 'trn':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs3D.size(0)
                running_dice += loss_dice.item() * inputs3D.size(0)
                count += inputs3D.size(0)

            epoch_loss = running_loss / count
            epoch_dice = running_dice / count

            print(f"  {phase} loss: {epoch_loss:.4f}, dice: {epoch_dice:.4f}")

            if phase == 'val' and epoch_dice < best_loss:
                best_loss = epoch_dice
                best_weights = copy.deepcopy(model.state_dict())

        scheduler.step()

    # Save best model
    model.load_state_dict(best_weights)
    torch.save(model.state_dict(), Path.home() / f"hybrid_lr{lr}_gamma{gamma}_bs{batch_size}_loss{best_loss}.pt")
    print("Training complete. Best dice loss:", best_loss)
    return best_loss

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

[I 2025-06-14 06:03:40,697] A new study created in memory with name: no-name-dce868f4-0c7f-4bc4-bb3b-4657ce235c6a


Starting training...
Epoch 01/30
  Training..  trn loss: 0.2817, dice: 0.6390
  Testing..  val loss: 0.3025, dice: 0.6747
Epoch 02/30
  Training..  trn loss: 0.1332, dice: 0.3411
  Testing..  val loss: 0.1934, dice: 0.4594
Epoch 03/30
  Training..  trn loss: 0.1200, dice: 0.2648
  Testing..  val loss: 0.1746, dice: 0.3649
Epoch 04/30
  Training..  trn loss: 0.1215, dice: 0.2515
  Testing..  val loss: 0.1206, dice: 0.2615
Epoch 05/30
  Training..  trn loss: 0.1195, dice: 0.2308
  Testing..  val loss: 0.3069, dice: 0.4857
Epoch 06/30
  Training..  trn loss: 0.1188, dice: 0.2166
  Testing..  val loss: 0.5874, dice: 0.7655
Epoch 07/30
  Training..  trn loss: 0.1171, dice: 0.2056
  Testing..  val loss: 0.1474, dice: 0.2713
Epoch 08/30
  Training..  trn loss: 0.1194, dice: 0.2001
  Testing..  val loss: 0.5530, dice: 0.7457
Epoch 09/30
  Training..  trn loss: 0.1264, dice: 0.2048
  Testing..  val loss: 0.2437, dice: 0.4042
Epoch 10/30
  Training..  trn loss: 0.1245, dice: 0.1950
  Testing..  

[I 2025-06-14 14:06:10,937] Trial 0 finished with value: 0.18310761987231672 and parameters: {'lr': 0.0015109001435323782, 'gamma': 0.9184014271851577}. Best is trial 0 with value: 0.18310761987231672.


Training complete. Best dice loss: 0.18310761987231672
Starting training...
Epoch 01/30
  Training..  trn loss: 82607.5852, dice: 0.7123
  Testing..  val loss: 0.4760, dice: 0.8306
Epoch 02/30
  Training..  trn loss: 178.0651, dice: 0.7535
  Testing..  val loss: 0.3039, dice: 0.7513
Epoch 03/30
  Training..  trn loss: 0.2712, dice: 0.6955
  Testing..  val loss: 0.3996, dice: 0.7074
Epoch 04/30
  Training..  trn loss: 0.2644, dice: 0.6303
  Testing..  val loss: 0.4357, dice: 0.6841
Epoch 05/30
  Training..  trn loss: 0.2608, dice: 0.5764
  Testing..  val loss: 0.5319, dice: 0.6896
Epoch 06/30
  Training..  trn loss: 0.2547, dice: 0.5260
  Testing..  val loss: 0.4923, dice: 0.6734
Epoch 07/30
  Training..  trn loss: 0.2542, dice: 0.4975
  Testing..  val loss: 0.5788, dice: 0.6833
Epoch 08/30
  Training..  trn loss: 0.2592, dice: 0.4827
  Testing..  val loss: 0.6174, dice: 0.6797
Epoch 09/30
  Training..  trn loss: 0.2636, dice: 0.4673
  Testing..  val loss: 0.7046, dice: 0.7037
Epoch 10/

[I 2025-06-14 22:02:14,375] Trial 1 finished with value: 0.6080518532544374 and parameters: {'lr': 0.014189472041726933, 'gamma': 0.8863361546461114}. Best is trial 0 with value: 0.18310761987231672.


Training complete. Best dice loss: 0.6080518532544374
Starting training...
Epoch 01/30
  Training..  trn loss: 127.6955, dice: 0.7160
  Testing..  val loss: 0.3633, dice: 0.7567
Epoch 02/30
  Training..  trn loss: 0.1825, dice: 0.5019
  Testing..  val loss: 3.1893, dice: 0.9987
Epoch 03/30
  Training..  trn loss: 0.1668, dice: 0.4202
  Testing..  val loss: 0.1947, dice: 0.4686
Epoch 04/30
  Training..  trn loss: 0.1589, dice: 0.3664
  Testing..  val loss: 0.5195, dice: 0.5998
Epoch 05/30
  Training..  trn loss: 0.1552, dice: 0.3337
  Testing..  val loss: 0.2875, dice: 0.4657
Epoch 06/30
  Training..  trn loss: 0.1543, dice: 0.3089
  Testing..  val loss: 0.2030, dice: 0.4195
Epoch 07/30
  Training..  trn loss: 0.1538, dice: 0.2919
  Testing..  val loss: 0.3131, dice: 0.4469
Epoch 08/30
  Training..  trn loss: 0.1539, dice: 0.2775
  Testing..  val loss: 0.3913, dice: 0.4892
Epoch 09/30
  Training..  trn loss: 0.1563, dice: 0.2673
  Testing..  val loss: 0.2527, dice: 0.4359
Epoch 10/30
  

[I 2025-06-15 06:07:17,776] Trial 2 finished with value: 0.29184270231053233 and parameters: {'lr': 0.004302606749291661, 'gamma': 0.9663563451763}. Best is trial 0 with value: 0.18310761987231672.


Training complete. Best dice loss: 0.29184270231053233
Starting training...
Epoch 01/30
  Training..  trn loss: 5356.6168, dice: 0.6797
  Testing..  val loss: 0.9614, dice: 0.8314
Epoch 02/30
  Training..  trn loss: 0.1614, dice: 0.4212
  Testing..  val loss: 0.1721, dice: 0.4495
Epoch 03/30
  Training..  trn loss: 0.1333, dice: 0.3022
  Testing..  val loss: 0.1707, dice: 0.4114
Epoch 04/30
  Training..  trn loss: 0.1260, dice: 0.2622
  Testing..  val loss: 0.1398, dice: 0.3256
Epoch 05/30
  Training..  trn loss: 0.1241, dice: 0.2406
  Testing..  val loss: 0.1377, dice: 0.2863
Epoch 06/30
  Training..  trn loss: 0.1240, dice: 0.2296
  Testing..  val loss: 0.1837, dice: 0.3379
Epoch 07/30
  Training..  trn loss: 0.1225, dice: 0.2158
  Testing..  val loss: 0.2209, dice: 0.4358
Epoch 08/30
  Training..  trn loss: 0.1239, dice: 0.2093
  Testing..  val loss: 0.1346, dice: 0.2352
Epoch 09/30
  Training..  trn loss: 0.1230, dice: 0.1992
  Testing..  val loss: 0.1348, dice: 0.2181
Epoch 10/30


In [ ]:
print("Best hyperparameters:", study.best_params)
print("Best dice loss:", study.best_value)